In [10]:
#imports 
import os
from typing import Optional
from pydantic import BaseModel

# LangChain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
# LangGraph
from langgraph.graph import StateGraph, END

In [11]:
policy_docs = [
    "Customers can return products within 7 days of delivery.",
    "Electronics items can only be returned within 3 days if defective.",
    "Clothing items must be unused and in original packaging.",
    "Shoes can be returned within 10 days if unused.",
    "Opened or used products are not eligible for full refund.",
    "Damaged products can be returned anytime within the return window.",
    "Refunds are issued only after quality check.",
    "Partial refunds are given for minor damages or missing packaging.",
    "Replacement is offered for size issues in clothing and shoes.",
    "Beauty and personal care items are non-returnable if opened."
]

In [12]:
#FAISS
# Convert to Documents
documents = [Document(page_content=doc) for doc in policy_docs]

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = splitter.split_documents(documents)

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Vector Store
vectorstore = FAISS.from_documents(docs, embeddings)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


C:\Users\User\AppData\Local\Temp\ipykernel_9476\3718341972.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
def query_policies(question: str) -> str:
    results = retriever.invoke(question)
    return "\n".join([doc.page_content for doc in results])

In [24]:
#statemodel
class ReturnState(BaseModel):
    user_input: str

    product_name: Optional[str] = None
    category: Optional[str] = None
    purchase_days_ago: Optional[int] = None
    reason: Optional[str] = None
    product_condition: Optional[str] = None

    retrieved_rules: Optional[str] = None
    return_status: Optional[str] = None
    resolution: Optional[str] = None
    final_response: Optional[str] = None

In [25]:
# tools
def resolution_tool(days: int, condition: str, category: str):
    # Default return window rules
    if category == "electronics":
        max_days = 3
    elif category == "shoes":
        max_days = 10
    else:
        max_days = 7

    # Condition check
    if days > max_days:
        return "Reject"

    if condition.lower() == "unused":
        if category in ["clothing", "shoes"]:
            return "Replacement"
        return "Refund"

    if condition.lower() == "damaged":
        return "Partial Refund"

    return "Reject"

In [26]:
#node 1 
import re

def extract_return_details(state: ReturnState):
    text = state.user_input.lower()

    # Days extraction
    days_match = re.search(r"(\d+)\s*days?", text)
    days = int(days_match.group(1)) if days_match else 0

    # Category detection
    if "shoe" in text:
        category = "shoes"
    elif "cloth" in text or "shirt" in text:
        category = "clothing"
    elif "phone" in text or "laptop" in text:
        category = "electronics"
    else:
        category = "general"

    # Condition
    if "unused" in text:
        condition = "unused"
    elif "damaged" in text:
        condition = "damaged"
    else:
        condition = "used"

    # Reason
    if "size" in text:
        reason = "size issue"
    else:
        reason = "other"

    state.purchase_days_ago = days
    state.category = category
    state.product_condition = condition
    state.reason = reason

    return state

In [27]:
#node 2
def retrieve_return_policy(state: ReturnState):
    query = f"{state.category} return policy {state.product_condition}"
    rules = query_policies(query)

    state.retrieved_rules = rules
    return state

In [28]:
#node 3
def validate_return(state: ReturnState):
    days = state.purchase_days_ago
    condition = state.product_condition
    category = state.category

    if category == "electronics" and days > 3:
        state.return_status = "Rejected"
    elif days > 10:
        state.return_status = "Rejected"
    elif condition == "used":
        state.return_status = "Conditional"
    else:
        state.return_status = "Eligible"

    return state

In [29]:
#node 4
def suggest_resolution(state: ReturnState):
    decision = resolution_tool(
        state.purchase_days_ago,
        state.product_condition,
        state.category
    )

    state.resolution = decision
    return state

In [30]:
#node 5
def generate_response(state: ReturnState):
    state.final_response = f"""
Return Status: {state.return_status}

Resolution: {state.resolution}

Reason:
- Category: {state.category}
- Days since purchase: {state.purchase_days_ago}
- Condition: {state.product_condition}
- Based on policies: {state.retrieved_rules}
"""
    return state

In [31]:
#workflow
workflow = StateGraph(ReturnState)

workflow.add_node("extract_return_details", extract_return_details)
workflow.add_node("retrieve_return_policy", retrieve_return_policy)
workflow.add_node("validate_return", validate_return)
workflow.add_node("suggest_resolution", suggest_resolution)
workflow.add_node("generate_response", generate_response)

workflow.add_edge("extract_return_details", "retrieve_return_policy")
workflow.add_edge("retrieve_return_policy", "validate_return")
workflow.add_edge("validate_return", "suggest_resolution")
workflow.add_edge("suggest_resolution", "generate_response")
workflow.add_edge("generate_response", END)

workflow.set_entry_point("extract_return_details")

return_app = workflow.compile()

## Test run

In [32]:
query = "I bought shoes 5 days ago, unused, want to return due to size issue"

result = return_app.invoke({
    "user_input": query
})

print(result["final_response"])


Return Status: Eligible

Resolution: Replacement

Reason:
- Category: shoes
- Days since purchase: 5
- Condition: unused
- Based on policies: Shoes can be returned within 10 days if unused.
Beauty and personal care items are non-returnable if opened.
Opened or used products are not eligible for full refund.

